In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load merged events

In [ ]:
def parse_list(value):
    if isinstance(value, list):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)) or str(value).strip() == "":
        return []
    try:
        parsed = json.loads(value)
    except (ValueError, TypeError):
        import ast
        parsed = ast.literal_eval(value)
    return parsed if isinstance(parsed, list) else [parsed]


merged_df = pd.read_csv(DATA_DIR / "merged_events_final.csv")
for c in ["article_ids", "source_event_ids", "source_texts", "actions", "subjects",
          "objects", "locations", "attributed_to", "quantities"]:
    merged_df[c] = merged_df[c].apply(parse_list)
for c in ["event_date", "period_end", "first_reported"]:
    merged_df[c] = pd.to_datetime(merged_df[c])
merged_df["event_time_label"] = merged_df["event_time_label"].fillna("")

articles_df = pd.read_csv(DATA_DIR / "articles_metadata.csv")
source_name = dict(zip(articles_df["article_id"], articles_df["source"]))

print("Merged events:", len(merged_df))
merged_df[["event_id", "event_date", "event_text", "article_ids"]].head()

## 2. Validate the dates
Every event date must lie close to the publication dates of the articles. Anything outside is flagged
instead of being silently placed on the timeline.

In [ ]:
pub_min = articles_df["publication_date"].astype("datetime64[ns]").min()
pub_max = articles_df["publication_date"].astype("datetime64[ns]").max()
lo, hi = pub_min - pd.Timedelta(days=7), pub_max + pd.Timedelta(days=7)

outside = merged_df[(merged_df["event_date"] < lo) | (merged_df["event_date"] > hi)]
print(f"Allowed window: {lo.date()} .. {hi.date()}")
print("Events outside the window:", len(outside))
if len(outside):
    display(outside[["event_id", "event_date", "event_text"]])
assert merged_df["event_date"].notna().all(), "some events have no date"

## 3. Order the events
1. by `event_date`
2. by time of day (clock time, or `morning`, `overnight`, ... when no clock time is given)
3. events **without** a time stay where the article tells them: they take the time of the closest earlier
   timed event of the same article and day (or, at the start of an article, of the next timed one),
   so "Water entered homes ..." is not pushed to the end of the day

In [ ]:
tl = merged_df.sort_values(["event_date", "rep_article", "rep_sentence", "rep_trigger"]).reset_index(drop=True)
grp = tl.groupby(["event_date", "rep_article"])["event_minutes"]
tl["order_minutes"] = grp.ffill()                                    # after the closest earlier timed event ...
lead_in = grp.bfill() - 0.5                                          # ... or just before the next timed one
tl["order_minutes"] = tl["order_minutes"].fillna(lead_in).fillna(-1) # nothing timed that day -> start of the day

tl = tl.sort_values(["event_date", "order_minutes", "rep_article", "rep_sentence", "rep_trigger"]).reset_index(drop=True)
tl.insert(0, "timeline_position", range(1, len(tl) + 1))

assert tl["event_date"].is_monotonic_increasing, "timeline is not chronological"
print("Timeline is chronological:", tl["event_date"].is_monotonic_increasing)
tl[["timeline_position", "event_date", "event_time_label", "event_text", "article_count"]]

## 4. Readable text for each entry

In [ ]:
DATE_BASIS = {
    "explicit_date": "date stated in the article",
    "weekday": "weekday stated in the article, resolved with the publication date",
    "relative_date": "relative date (yesterday / today ...) resolved with the publication date",
    "inherited_from_parent_event": "same date as the other event of that sentence",
    "publication_date_with_time": "time stated, day taken from the publication date",
    "publication_date": "no date in the text - publication date used",
}


def as_sentence(text):
    text = text.strip()
    return text if text.endswith((".", "!", "?")) else text + "."


def describe_sources(ids):
    return ", ".join(f"{a} ({source_name.get(a, 'unknown source')})" for a in ids)


tl["date_label"] = tl["event_date"].dt.strftime("%Y-%m-%d (%A)")
tl["location_text"] = tl["locations"].apply(lambda l: ", ".join(l) if l else "Not mentioned")
tl["statement_by"] = tl["attributed_to"].apply(lambda l: ", ".join(l))
tl["date_basis"] = tl["date_source"].map(DATE_BASIS).fillna(tl["date_source"])
tl["period_until"] = tl["period_end"].dt.strftime("%Y-%m-%d (%A)")

## 5. Write the timeline

In [ ]:
txt_path = OUTPUT_DIR / "final_news_timeline_v2.txt"

with open(txt_path, "w", encoding="utf-8") as f:
    f.write("=" * 90 + "\n")
    f.write("                  NEWS EVENT TIMELINE\n")
    f.write("=" * 90 + "\n")
    f.write(f"{len(tl)} events from {articles_df['article_id'].nunique()} articles\n")

    current_day = None
    for _, r in tl.iterrows():
        if r["event_date"] != current_day:
            current_day = r["event_date"]
            f.write("\n" + "#" * 90 + "\n")
            f.write(f"# {r['event_date'].strftime('%A, %d %B %Y')}\n")
            f.write("#" * 90 + "\n\n")

        f.write(f"DATE: {r['date_label']}\n")
        if r["event_time_label"]:
            f.write(f"TIME: {r['event_time_label']}\n")
        f.write(f"EVENT ID: {r['event_id']}\n")
        f.write(f"EVENT: {as_sentence(r['event_text'])}\n")
        if r["statement_by"]:
            f.write(f"STATED BY: {r['statement_by']}\n")
        f.write(f"LOCATION: {r['location_text']}\n")
        if pd.notna(r["period_end"]):
            f.write(f"COVERS PERIOD UNTIL: {r['period_until']}\n")
        f.write(f"REPORTED BY: {r['article_count']} article(s)\n")
        f.write(f"SOURCES: {describe_sources(r['article_ids'])}\n")
        if r["article_count"] > 1:
            for line in r["source_texts"]:
                f.write(f"   - {line}\n")
        f.write(f"DATE BASIS: {r['date_basis']}\n")
        f.write("-" * 90 + "\n\n")

print("Timeline saved to", txt_path)

In [ ]:
csv_cols = ["timeline_position", "event_id", "event_date", "event_time_label", "event_text", "statement_by",
            "location_text", "period_end", "article_count", "article_ids", "date_confidence", "date_source"]
csv_out = tl[csv_cols].copy()
csv_out["event_date"] = csv_out["event_date"].dt.strftime("%Y-%m-%d")
csv_out["period_end"] = csv_out["period_end"].dt.strftime("%Y-%m-%d")
csv_out["article_ids"] = csv_out["article_ids"].apply(lambda x: ", ".join(x))
csv_out.to_csv(OUTPUT_DIR / "final_news_timeline_v2.csv", index=False)
print("Timeline saved to", OUTPUT_DIR / "final_news_timeline_v2.csv")
csv_out.head(40)

## 6. Preview

In [ ]:
print(open(txt_path, encoding="utf-8").read()[:4500])